# Scikit-Fuzzy

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Implementar** funciones de membresía con `scikit-fuzzy` (`fuzz.trimf`, `fuzz.gaussmf`, …).
2. **Construir** un sistema de control difuso con `ctrl.Antecedent`, `ctrl.Consequent` y `ctrl.ControlSystem`.
3. **Definir** el conjunto de reglas difusas y simular el sistema con `ctrl.ControlSystemSimulation`.
4. **Analizar** la superficie de control y la sensibilidad del sistema al diseño de las reglas.
5. **Conectar** la teoría del notebook `02_SistemasDifusos` con su implementación práctica.

> 🧪 **Laboratorio:** este cuaderno es la contraparte práctica del módulo de Sistemas Difusos.

> 💡 **Prerrequisitos:** [02_SistemasDifusos](02_SistemasDifusos.ipynb).

- Biblioteca Python desarrollada para implementar sistemas de inferencia difusos sobre el stack de la biblioteca SciPy

# Clase Práctica SciKit-Fuzzy

## Ejemplo 1: Propina

Ud. acude al restaurant **El maestro difuso**, el rey de la comida univesitaria cercano a la **Universidad Difusa Federico Santa María**. Ud. como buen comensal, desea dejar propina, pero su decisión estará basada en la calidad de la comida y la atención del mesero: Si las cosas van mal, no dejará propina y si van de forma inigualable dejará hasta 15%, vale decir, añadirá hasta un 5% extra al 10% sugerido. El resto variará entre una fracción del 10% base y el sugerido normalmente.

Ahora, considerando el común nacional, las evaluaciones se hacen en una escala del 1 al 7 con décimas. Sin embargo, ud. define sus escala subjetivas de la siguientes formas ascendentes:
* Comida: `pesimo`, `malito`, `piola`, `de pana`, `basadísimo`.
* Atención: `amarga`, `pesada`, `noesni`, `tela`, `joya`.
Además, no tiene claro que valor de la escala representa cada una de sus calificaciones. 

**Construya un sistema de lógica difusa que le permita resolver cuanta propina debiese dejar según la probable atención en el restaurant**. Defina claramente variables numéricas, difusas, funciones de pertenencia, reglas difusas. Además explique, ¿Cuándo ocurre la Fuzzificación y la Defuzzificación?

<center><img src="images/fuzzylogic_system.png" align="center"/></center>

<center><img src="images/fuzzy_steps.png" align="center"/></center>

# Variables

### Variables de Entradas:
* ¿Qué tan bueno estuvo el almuerzo?:
    * Variable: `comida`.
    * Como variable numérica, universo: $\{1.0, 1.1, 1.2, 1.3,\dots,6.7, 6.8, 6.9, 7.0\}$
    * Como variable difusa: $\{\texttt{pesimo}, \texttt{ malito}, \texttt{ piola}, \texttt{ de pana}, \texttt{ basadísimo}\}$

* ¿Qué tan bien te atendieron?:
    * Variable: `atención`.
    * Como variable numérica, universo: $\{1.0, 1.1, 1.2, 1.3,\dots,6.7, 6.8, 6.9, 7.0\}$
    * Como variable difusa: $\{\texttt{amarga}, \texttt{ pesada}, \texttt{ noesni}, \texttt{ tela}, \texttt{ joya}\}$

### Variables de Salida
* ¿Cuánta propina voy a dejar?
    * Variable: `propina`
    * Como variable numérica, universo (en %): $\{0, 1, 2, 3, \dots, 9, 10, 11, 12, 13, 14, 15\}$
    * Como variable difusa: $\{\texttt{nada}, \texttt{ poco}, \texttt{ normal}, \texttt{ extra}\}$

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz

from skfuzzy import control as ctrl

In [ ]:
# Variable de Entradas o Antecedentes
calidad = ctrl.Antecedent(np.arange(1, 7.1, 0.1), 'calidad') #pésima, malita, piola, de pana, basadísimo
atención = ctrl.Antecedent(np.arange(1, 7.1, 0.1), 'atención') #amarga, pesada, noesni, tela, joya

#Variable de salida o Consecuentes
propina = ctrl.Consequent(np.arange(0, 15.1, 0.1), 'propina')

#.universe es un método para Antecedentes y Consecuentes, que entrega el dominio de la variable numérica
print(propina.universe) 

# Funciones de Membresía

In [ ]:
# Definir funciones de membresía
# Cada valor de la variable difusa debe tener una función de membresía
# funciones de referencia en: https://pythonhosted.org/scikit-fuzzy/api/skfuzzy.membership.html
calidad['pésima'] = fuzz.gaussmf(calidad.universe, 1,0.5)
calidad['malita'] = fuzz.trimf(calidad.universe,[1.5,3,4.5])
calidad['piola'] = fuzz.pimf(calidad.universe, 3,4,5,6)
calidad['de pana'] = fuzz.trapmf(calidad.universe,[5,5.5,6.0,6.5])
calidad['basadísimo'] = fuzz.smf(calidad.universe,6.0,7)

#graficar membresías específica utilizando el operador ['categoria'], i.e., calidad['pesimo'].view()
calidad.view()
calidad["pésima"].view()

In [ ]:
# Existe un método de generación automática de función de membresía triangular
# Funciona solo con los valores 3, 5, o 7.
# Para otra cantidad de categorías, se debe hacer manualmente o pasar una lista con n labels 
nombres = ["amarga", "pesada", "noesni", "tela", "joya"]
atención.automf(names=nombres)
atención.view()

In [ ]:
# Consecuente y funciones de pertenencia
salida=["nada", "poca", "normal", "extra"]
propina.automf(names=salida) #comenzamos con automático
propina["nada"]=fuzz.zmf(propina.universe,0,1) #sobreescribimos algunas
propina["poca"]=fuzz.gaussmf(propina.universe, 5, 1.25)
propina["extra"]=fuzz.smf(propina.universe,10,15)
propina.view()

# Reglas Difusas

```Si las cosas van mal, no dejará propina y si van de forma inigualable dejará hasta 15%, vale decir, añadirá hasta un 5% extra al 10% sugerido. El resto variará entre una fracción del 10% base y el sugerido normalmente.```

* **Sí** la calidad de la comida es _pésima_ y la atención del mesero es _amarga_, **entonces** la propina será _nada_.
* **Sí** la calidad de la comida es _malita_ o _mas o menos_, **entonces** la propina será _poca_.
* **Sí** la atención del mesero es _pesada_ o _noesni_, **entonces** la propina será _poca_.
* **Sí** la calidad de la comida es _de pana_, **entonces** la propina será _normal_.
* **Sí** la atención del mesero es _tela_, **entonces** la propina será _normal_.
* **Sí** la calidad de la comida es _basadísima_ y la atención del mesero es _joya_, **entonces** la propina será _extra_.


In [ ]:
# Cada regla relaciona antecedentes con consecuentes. Los resuelve segun su valor de la función de membresía.
regla1 = ctrl.Rule(calidad['pésima'] & atención['amarga'], propina['nada'])
regla2 = ctrl.Rule(calidad['malita'] | calidad['piola'], propina['poca'])
regla3 = ctrl.Rule(atención['pesada'] | atención['noesni'], propina['poca'])
regla4_5 = ctrl.Rule((calidad['de pana'] | atención['tela']), propina['normal'])
regla6 = ctrl.Rule(calidad['basadísimo'] & atención['joya'], propina['extra'])

#pésima, malita, piola, de pana, basadísimo
# View permite ver la complejidad del tema, donde los nodos de termino son los posibles valores que puede tomar el consecuente.
# regla6.view() no funciona en esta versión(?)

In [ ]:
#Creamos un Systema de Control Difuso, el cual combina todas las reglas creadas.
control_propina = ctrl.ControlSystem([regla1, regla2, regla3, regla4_5, regla6])

In [ ]:
#Creamos una Simulación del Sistema de Control Difuso. 
dejar_propina = ctrl.ControlSystemSimulation(control_propina)

#A esta simulación, le podemos asignar, según nombre de los antecedentes, un valor de entrada númerico.
dejar_propina.input['calidad'] = 4.32
dejar_propina.input['atención'] = 5.123

# Luego, podemos computar según el proceso el Sistema Difuso creado
dejar_propina.compute()


In [ ]:
#Podemos imprimir la salida numérica (desfuzzificada)

print("\nLa propina debe ser de un", str(dejar_propina.output['propina'])+"%\n")


In [ ]:
#Podemos ver los graficos de las funciones de membresía sobre los valores de los antecedentes y consecuentes. 
calidad.view(sim=dejar_propina)
atención.view(sim=dejar_propina)
propina.view(sim=dejar_propina)

## Ejercicio 1: Aire Acondicionado. 

Ud. ha sido contratado por IPD-Electrónics para trabajar en su marca TEL-Air que comercializará equipos de aire acondicionado. Estos dispositivos climatizan ambientes interiores en el rango entre $17ºC$ a $30ºC$, lanzando aire frio o aire caliente dependiendo de las instrucciones que el usuario le entregue, hasta llegar a la temperatura deseada. Así:

* Para climatizar utilizando el modo _calor_, el aire acondicionado lanza aire caliente constante, a unos $42ºC$, hasta llegar a la temperatura deseada. 
* Para climatizar utilizando el modo _frío_, el aire acondicionado lanza aire fresco constante, a unos $13ºC$, hasta llegar a la temperatura deseada.
* Para mantener el clima actual del ambiente se utiliza el modo _stand by_, que provee de circulación del aire sin modificar la variación normal de la temperatura.
* Para climatizar automáticamente se utiliza el modo _auto_, el cual se encarga de escoger el modo del aire acondicionado, y el nivel de velocidad utilizado.

El aire es lanzado por los dispositivos en tres posibles velocidades: baja, media o alta. Estas velocidades modifican la temperatura, según el modo de aire y las caractiristicas de la zona climatizada, en los rangos (en $\frac{ºC}{\text{min}}$) $[0.1, 0.7]$, $[0.5, 1.2]$ y $[0.8, 1.9]$, respectivamente. Considere que si está en modo _stand by_ estas variaciones se darán para acercarse a la temperatura del exterior.  

Para controlar la temperatura del ambiente, los dispositivos cuentan con un termostato que les indica la temperatura actual de la zona climatizada.

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, no puede utilizar el modo de velocidad alta.
* Si se encuentra muy alejado en la tempratura, debe utilizar el modo de velocidad alta. 

Y que además, utilizando el historico de temperatura actual y de destino, reflejado en las configuraciones manuales, se ha determinado qué:
* La temperatura puede ser catalogada como fría si está bajo el umbral de los 15ºC.
* La temperatura puede ser catalogada como fresca si está en el rango de los 12.5ºC a los 20ºC.
* La temperatura puede ser catalogada como ambiente entre los 18ºC a los 25ºC.
* La temperatura puede ser catalogada como calurosa sobre el rango de los 23ºC.

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

1. Genere un sistema de inferencia difusa que permita para un instante determinado, utilizando el modo _auto_, determinar el modo de función del dispositivo y la velocidad con la cual lanza el aire. Determine claramente los antecendentes y los consecuentes del sistema, funciones de membresía y las reglas de inferencia difusa del sistema.

In [ ]:
# Incluir librerías
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

## Variables
### Antecedentes

In [ ]:
amb = ctrl.Antecedent(np.arange(0, 40.5, 0.5), 'ambiente') #frio, fresco, ambiente, caluroso
dif = ctrl.Antecedent(np.arange(-30, 23.5, 0.5), 'diferencia') #muy negativo, negativo, neutro, positivo, muy positivo

### Consecuentes

In [ ]:
vel = ctrl.Consequent(np.arange(0.1, 2.0, 0.1), "velocidad") # baja, media, alta
modo = ctrl.Consequent(np.arange(-1,1.01, 0.01), "modo") #frio, stand by, calor

# Membresías

In [ ]:
amb['frio'] = fuzz.zmf(amb.universe, 0,15)
amb['fresco'] = fuzz.pimf(amb.universe, 12.5, 15, 18, 20)
amb['ambiente'] = fuzz.gaussmf(amb.universe, 21, 1.5)
amb['caluroso'] = fuzz.smf(amb.universe, 23, 40)
amb.view()

In [ ]:
diffs = ['muy negativo', 'negativo', 'neutro', 'positivo', 'muy positivo']
dif.automf(names=diffs)
dif.view()

In [ ]:
vel['baja'] = fuzz.pimf(vel.universe, 0.0, 0.15, 0.65, 0.8)
vel['media'] = fuzz.pimf(vel.universe, 0.3, 0.45, 1.15, 1.3)
vel['alta'] = fuzz.pimf(vel.universe, 0.7, 0.85, 1.85, 2.0)
vel.view()

In [ ]:
modo['frio'] = fuzz.zmf(modo.universe,-1,0)
modo['stand by'] = fuzz.gaussmf(modo.universe, 0, 0.1) 
modo['calor'] = fuzz.smf(modo.universe, 0, 1)
modo.view()

Considere que para limitar el consumo energético, el modo _auto_ define las siguientes reglas bases:
* Si la diferencia de temperatura es negativa, debe utilizar modo calor.
* Si la diferencia de temperatura es positiva, debe utilizar modo frío.
* Si no hay diferencia de temperatura, debe utilizar el modo stand by.
* Si se encuentra cercano o en la temperatura deseada, no puede utilizar el modo de velocidad alta.
* Si se encuentra muy alejado en la tempratura, debe utilizar el modo de velocidad alta. 

Y como es un sistema basado en la percepción y el comfort humano, se definen las siguientes reglas básicas:
* Si la temperatura es fría, el aire debe utilizar modo calor. 
* Si la temperatura es calurosa, el aire debe utilizar modo frío.

In [ ]:
r1 = ctrl.Rule(dif['negativo'] | dif['muy negativo'], modo['calor'])
r2 = ctrl.Rule(dif['positivo'] | dif['muy positivo'], modo['frio'])
r3 = ctrl.Rule(dif['neutro'], modo['stand by'])
r4_1 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['baja'])
r4_2 = ctrl.Rule(dif['neutro'] | dif['negativo'] | dif['positivo'], vel['media'])
r5 = ctrl.Rule(dif['muy negativo'] | dif['muy positivo'], vel['alta'])

r6 = ctrl.Rule(amb['frio'], modo['calor'])
r7 = ctrl.Rule(amb['caluroso'], modo['frio'])

ctrl_aire = ctrl.ControlSystem([r1, r2, r3, r4_1, r4_2, r5, r6, r7])

In [ ]:
AC = ctrl.ControlSystemSimulation(ctrl_aire)

t_ambiente = 20.0
t_destino = 17.0

AC.input['ambiente'] = t_ambiente
AC.input['diferencia'] = t_ambiente - t_destino

AC.compute()

In [ ]:
print(AC.output['modo'])
print(AC.output['velocidad'])
dif.view(sim=AC)
amb.view(sim=AC)
modo.view(sim=AC)
vel.view(sim=AC)

2. Simule un sistema de control difuso, que muestre como funcionaría el dispositivo durante 15 minutos, considerando una actualización de las condiciones cada 30 segundos.